# DAY1 실습 01-1 — 데이터 구조와 범주형 기술통계 — 정답

이 노트북은 **실습 문제만** 담고 있다. 개념과 기본 문법은 PPT 「DAY1 기술통계와 데이터 분포」에서 확인한다.
각 문제 아래의 `PPT p.번호`가 참고할 슬라이드이다. 빈 코드 셀에 답을 작성하고 실행한다.

In [ ]:
%pip install -q koreanize-matplotlib   # 한글 폰트 자동 설정. Colab은 세션마다 다시 설치한다

import numpy as np
import pandas as pd
import koreanize_matplotlib

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "ai4i2020.csv")

df = pd.read_csv(csv_path)   # 원본
data = df.copy()             # 실습용 사본

### 문제 1. 데이터 구조 보기
`PPT p.23`

`data`의 처음 5행, 행·열 수, 열 이름 목록을 출력하시오.

**확인** : (10000, 14)

In [ ]:
display(data.head())
print("행·열:", data.shape)
print("열 이름:", data.columns.tolist())

### 문제 2. 저장 자료형과 통계적 유형
`PPT p.23, 22, 18`

`info()`로 열별 저장 자료형을 확인하시오. 그다음 아래 네 변수의 **통계적 유형**을 주석으로 적으시오.

`UDI` · `Type` · `Torque [Nm]` · `Machine failure`

In [ ]:
data.info()

# UDI             : 식별자 (int64지만 평균을 내지 않는다)
# Type            : 명목형 범주형 (L·M·H)
# Torque [Nm]     : 수치형 (연속형 측정값)
# Machine failure : 이진 범주형 (0=정상, 1=고장)

### 문제 3. 결측값과 완전 중복 행
`PPT p.24`

열별 결측값 수와 완전 중복 행 수를 구하시오. 이어서 `Torque [Nm]` 열만의 결측값 수도 구하시오.

**확인** : 결측 0개, 중복 0개

In [ ]:
display(data.isna().sum().rename("결측값 수"))
print("완전 중복 행:", data.duplicated().sum())
print("Torque 결측값:", data["Torque [Nm]"].isna().sum())

### 문제 4. Type 범주별 빈도
`PPT p.29`

`Type`의 범주별 빈도를 구해 `type_cnt`에 담고 출력하시오.

**확인** : L 6000, M 2997, H 1003

In [ ]:
type_cnt = data["Type"].value_counts()
type_cnt

### 문제 5. Type 비율·백분율 표
`PPT p.29`

`Type`의 비율과 백분율(%)을 열로 갖는 표 `df_rate`를 만들고 소수 넷째 자리로 반올림하시오.
행 이름은 L → 저품질(L), M → 중품질(M), H → 고품질(H)로 바꾸고, 비율 계산의 분모(유효값 수)도 출력하시오.

**확인** : 저품질(L) 0.6000 / 60.00, 분모 10000

In [ ]:
type_rate = data["Type"].value_counts(normalize=True)
df_rate = pd.DataFrame({"비율": type_rate, "백분율(%)": type_rate * 100}).round(4)
df_rate = df_rate.rename(index={"L": "저품질(L)", "M": "중품질(M)", "H": "고품질(H)"})
display(df_rate)
print("분모:", data["Type"].count())

### 문제 6. Type 최빈값
`PPT p.31`

`Type`에서 가장 자주 나타나는 범주(최빈값)를 구하시오.

**확인** : L

In [ ]:
data["Type"].mode()

### 문제 7. Machine failure 빈도와 최빈값
`PPT p.31, 32`

`Machine failure` 열의 빈도와 최빈값을 구하시오.

**확인** : 0 → 9661건, 1 → 339건, 최빈값 0

In [ ]:
print(data["Machine failure"].value_counts())
print("최빈값:", data["Machine failure"].mode()[0])

In [ ]:
# [데이터 준비] 문제 8에 사용한다. 그대로 실행한다
status = pd.Series(["A", "A", "B", np.nan], name="status")

### 문제 8. 결측값이 있을 때의 분모
`PPT p.30`

`status`에서 A의 개수를 센 뒤, **전체 행 기준**(`len`)과 **유효값 기준**(`count`)으로 A의 비율을 각각 구해 표 `comp`로 정리하시오.
열 이름은 `기준`, `분자_A빈도`, `분모`, `A비율`로 한다.

**확인** : 전체 행 0.50, 유효값 0.67

In [ ]:
cnt = status.eq("A").sum()
n_all, n_valid = len(status), status.count()
comp = pd.DataFrame({
    "기준": ["전체 행", "유효값"],
    "분자_A빈도": [cnt, cnt],
    "분모": [n_all, n_valid],
    "A비율": [cnt / n_all, cnt / n_valid],
})
comp

In [ ]:
# [데이터 준비] 문제 9에 사용한다. 그대로 실행한다
failure = pd.Series([0, 1, 0, None, 1])   # 0=정상, 1=고장, None=결측

### 문제 9. 0/1 변수의 평균과 고장률
`PPT p.32, 33`

`failure`의 전체 건수, 유효 관측수, 고장 건수를 출력하시오.
전체 기준 고장률과 유효값 기준 고장률을 각각 계산하고, `mean()` 결과와 비교하시오.
마지막으로 결측까지 포함한 값별 건수(`dropna=False`)를 값 순서로 출력하시오.

**확인** : 전체 기준 0.4, 유효값 기준 0.5, mean() 0.5

In [ ]:
fail_count = (failure == 1).sum()
print("전체 건수:", len(failure), "/ 유효 관측수:", failure.count(), "/ 고장 건수:", fail_count)
print("전체 기준 고장률:", fail_count / len(failure))
print("유효값 기준 고장률:", fail_count / failure.count())
print("mean():", failure.mean())
print(failure.value_counts(dropna=False).sort_index())

### 문제 10. Machine failure 빈도·비율·백분율 표
`PPT p.32`

`Machine failure`의 0(정상)과 1(고장)에 대해 빈도, 비율, 백분율(%)을 열로 갖는 표 `result`를 만드시오.
행 이름은 0 → 정상, 1 → 고장으로 바꾼다.

**확인** : 고장 339건, 비율 0.0339, 3.39%

In [ ]:
cnt = data["Machine failure"].value_counts().sort_index()
rate = data["Machine failure"].value_counts(normalize=True).sort_index()
result = pd.DataFrame({"빈도": cnt, "비율": rate, "백분율(%)": rate * 100})
result = result.rename(index={0: "정상", 1: "고장"})
result

### 문제 11. HDF 빈도와 비율
`PPT p.32`

`HDF`(열 방출 고장)의 0과 1에 대해 `빈도`, `비율` 두 열로 된 표를 0, 1 순서로 출력하시오.

**확인** : 0 → 9,885건, 1 → 115건

In [ ]:
hdf = pd.DataFrame({"빈도": data["HDF"].value_counts(), "비율": data["HDF"].value_counts(normalize=True)}).sort_index()
hdf

### 문제 12. TWF 고장 건수와 백분율
`PPT p.32`

`TWF`(공구 마모 고장)에서 1의 건수와 고장 백분율(%)을 소수 둘째 자리까지 출력하시오. 백분율은 0/1 변수의 평균을 이용한다.

**확인** : 46건, 0.46%

In [ ]:
print("고장 건수:", (data["TWF"] == 1).sum())
print("고장 백분율(%):", round(data["TWF"].mean() * 100, 2))

## 종합 실습 : 결측과 중복이 있는 점검 기록

| 열 | 뜻 |
|---|---|
| 설비ID | 설비 번호 |
| 라인 | 생산 라인 (A·B·C) |
| 교대 | 주간·야간 |
| 고장 | 0 = 정상, 1 = 고장 |

중복 처리 절차(탐지 → 원인 판단 → 제거 → 검증 → 집계)는 `PPT p.25`를 참고한다.

In [ ]:
# [데이터 준비] 그대로 실행한다
records = pd.DataFrame({
    "설비ID": ["E01", "E02", "E03", "E04", "E05", "E06", "E07", "E08", "E09", "E06"],
    "라인":   ["A",   "A",   "B",   "B",   "C",   "A",   "B",   "C",   np.nan, "A"],
    "교대":   ["주간", "야간", "주간", "야간", "주간", "주간", "야간", "야간", "주간", "주간"],
    "고장":   [0,     1,     0,     np.nan, 1,    0,     0,     np.nan, 1,     0],
})
records

### 문제 종합 1. 크기와 자료형
`PPT p.23`

몇 행 몇 열이고, 각 열의 저장 자료형은 무엇인가?

In [ ]:
print(records.shape)
print(records.dtypes)

### 문제 종합 2. 결측과 중복
`PPT p.24`

열별 결측값 수와 완전 중복 행 수는 몇 개인가?

**확인** : 라인 1개, 고장 2개 결측 / 중복 1행

In [ ]:
display(records.isna().sum())
print("완전 중복 행:", records.duplicated().sum())

### 문제 종합 3. 중복 확인과 제거
`PPT p.25`

중복된 행을 모두 출력해 보고, 중복을 제거한 표를 `check`로 저장한 뒤 행 수와 중복 수를 다시 확인하시오.
(힌트 : `duplicated(keep=False)`)

**확인** : 10행 → 9행, 중복 0

In [ ]:
display(records[records.duplicated(keep=False)])
check = records.drop_duplicates()
print("제거 전:", len(records), "→ 제거 후:", len(check))
print("제거 후 중복:", check.duplicated().sum())

In [ ]:
# [데이터 준비] 종합 3을 풀지 못한 경우 이 셀을 실행한 뒤 종합 4~6을 푼다
check = records.drop_duplicates()

### 문제 종합 4. 중복 제거 전후 비교
`PPT p.29`

라인별 빈도와 비율(유효값 기준)을 중복 제거 전(`records`)과 후(`check`)로 나란히 놓은 표를 만들고 비교하시오.

In [ ]:
compare = pd.DataFrame({
    "빈도_전": records["라인"].value_counts(), "비율_전": records["라인"].value_counts(normalize=True).round(3),
    "빈도_후": check["라인"].value_counts(),   "비율_후": check["라인"].value_counts(normalize=True).round(3),
})
compare

### 문제 종합 5. 다중 최빈값
`PPT p.31`

중복을 제거한 표(`check`)에서 라인의 최빈값은 무엇인가? 결과가 몇 개 나오며 왜 그런가?

**확인** : A, B 두 개

In [ ]:
print(check["라인"].value_counts())
print(check["라인"].mode())
# A와 B가 3번씩으로 가장 많이 나와 빈도가 같다 → 최빈값이 2개

### 문제 종합 6. 두 기준의 고장률
`PPT p.30, 32`

중복을 제거한 표(`check`)에서 고장률을 전체 행 기준과 유효값 기준으로 각각 구하시오.

**확인** : 3 ÷ 9 = 0.3333, 3 ÷ 7 = 0.4286

In [ ]:
fail = (check["고장"] == 1).sum()
print("전체 행 기준:", round(fail / len(check), 4))
print("유효값 기준:", round(fail / check["고장"].count(), 4), "= mean()", round(check["고장"].mean(), 4))